# Phase 4: TAC-LAnoBERT Experiments (E1-E3)

**Prerequisites**:
- ✅ Phase 2: LAnoBERT baseline trained → `outputs/BGL_lanobert/`
- ✅ Phase 3: TAC-LAnoBERT trained → `outputs/BGL_tac/`
- ✅ BGL data preprocessed → `data/BGL/`

**What this notebook does**:
- **E1**: Load & verify baseline metrics (from Phase 2)
- **E2**: Compare TAC-LAnoBERT vs baseline
- **E3**: Measure early detection capability (DLT, EWR)

## Setup

In [ ]:
!git clone https://github.com/rubyhcm/TAC-LAnoBERT.git
%cd TAC-LAnoBERT

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
# Verify environment
import torch
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

## Check Prerequisites

In [ ]:
# Check Phase 2/3 outputs and BGL data (copy from Kaggle datasets)
import glob

print("=" * 70)
print("CHECKING PREREQUISITES")
print("=" * 70)

# Phase 2 baseline
baseline_in_input = glob.glob("/kaggle/input/**/BGL_lanobert", recursive=True)
if os.path.exists("outputs/BGL_lanobert/model/final"):
    print("\n✅ Phase 2 baseline: Already in working directory")
elif baseline_in_input:
    print(f"\n📦 Phase 2 baseline: Found in input, copying...")
    os.makedirs("outputs", exist_ok=True)
    !cp -r {baseline_in_input[0]} outputs/
    print("✅ Copied")
else:
    raise FileNotFoundError("❌ Phase 2 baseline not found. Upload as Kaggle dataset.")

# Phase 3 TAC
tac_in_input = glob.glob("/kaggle/input/**/BGL_tac", recursive=True)
if os.path.exists("outputs/BGL_tac/model/final"):
    print("\n✅ Phase 3 TAC: Already in working directory")
elif tac_in_input:
    print(f"\n📦 Phase 3 TAC: Found in input, copying...")
    os.makedirs("outputs", exist_ok=True)
    !cp -r {tac_in_input[0]} outputs/
    print("✅ Copied")
else:
    raise FileNotFoundError("❌ Phase 3 TAC not found. Upload as Kaggle dataset.")

# BGL preprocessed data
bgl_data_in_input = glob.glob("/kaggle/input/**/BGL/", recursive=True)
if all(os.path.exists(f"data/BGL/{f}") for f in ["BGL_test_parsed.log", "BGL_test_label.log"]):
    print("\n✅ BGL data: Already preprocessed")
elif bgl_data_in_input:
    print(f"\n📦 BGL data: Found in input, copying...")
    os.makedirs("data/BGL", exist_ok=True)
    !cp -rn {bgl_data_in_input[0]}/. data/BGL/
    print("✅ Copied")
else:
    print("\n⚠️  BGL preprocessed data not found. Will split from raw BGL.log...")
    bgl_logs = glob.glob("/kaggle/input/**/BGL.log", recursive=True)
    if bgl_logs:
        os.makedirs("data/BGL", exist_ok=True)
        !ln -sf {bgl_logs[0]} data/BGL/BGL.log
        !python -m tac_lanobert.split_tac --config configs/bgl_tac_full.yaml
        print("✅ Data split complete")
    else:
        raise FileNotFoundError("❌ BGL.log not found")

print("\n" + "=" * 70)
print("✅ ALL PREREQUISITES READY")
print("=" * 70)

## E1: Baseline Metrics Verification

Load Phase 2 baseline results and verify against paper targets.

In [ ]:
!python experiments/run_phase4.py --experiment E1

## E2: TAC-LAnoBERT Inference + Comparison

Run TAC inference (if not already done) and compare with baseline.

In [ ]:
# Check if TAC inference already done
import os

tac_scores_exist = os.path.exists("outputs/BGL_tac/results/scores_tac_hybrid.npy")

if not tac_scores_exist:
    print("Running TAC inference (Memory Queue + Hybrid Scoring)...")
    print("This will take ~2 hours on T4 GPU\n")
    !python -m tac_lanobert.inference_tac --config configs/bgl_tac_full.yaml
else:
    print("✅ TAC inference already complete (scores found)")

In [ ]:
!python experiments/run_phase4.py --experiment E2

## E3: Early Detection Test (DLT, EWR)

Measure Detection Lead Time and Early Warning Rate.

In [ ]:
!python experiments/run_phase4.py --experiment E3

## Summary

View all results:

In [ ]:
import json

print("=" * 70)
print("PHASE 4 RESULTS SUMMARY")
print("=" * 70)

# E1
if os.path.exists("outputs/phase4_e1_baseline_reference.json"):
    with open("outputs/phase4_e1_baseline_reference.json") as f:
        e1 = json.load(f)
    print(f"\n📊 E1 - Baseline:")
    # E1 only has score stats, not full metrics
    print(f"   Lines:  {e1.get('num_lines', 'N/A'):,}")
    print(f"   Mean Score: {e1.get('score_mean', 0):.6f}")

# E2
if os.path.exists("outputs/phase4_e2_comparison_report.json"):
    with open("outputs/phase4_e2_comparison_report.json") as f:
        e2 = json.load(f)
    print(f"\n📊 E2 - TAC vs Baseline:")
    print(f"   Baseline Mean:   {e2.get('baseline_mean', 0):.6f}")
    print(f"   TAC Hybrid Mean: {e2.get('tac_hybrid_mean', 0):.6f}")
    diff = e2.get('diff_pct', 0)
    print(f"   Difference:      {diff:+.2f}%")

# E3
if os.path.exists("outputs/phase4_e3_early_detection_report.json"):
    with open("outputs/phase4_e3_early_detection_report.json") as f:
        e3 = json.load(f)
    print(f"\n📊 E3 - Early Detection:")
    print(f"   Mean DLT:      {e3.get('mean_dlt_seconds', 0) / 60:.2f} minutes")
    print(f"   Median DLT:    {e3.get('median_dlt_seconds', 0) / 60:.2f} minutes")
    print(f"   Max DLT:       {e3.get('max_dlt_seconds', 0) / 60:.2f} minutes")
    print(f"   EWR (≥5 min):  {e3.get('ewr_pct', 0):.2f}%")
    print(f"   DLT > 0:       {e3.get('dlt_positive_pct', 0):.2f}%")

print("\n" + "=" * 70)
print("✅ Phase 4 Complete")
print("=" * 70)